In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [2]:
# Load CSV
try:
    df = pd.read_csv("../../Datasets/csv/UAVUnderNormalCyberattacks.csv", low_memory=False)
except FileNotFoundError:
    print("Error: The file '../../Datasets/csv/UAVUnderNormalCyberattacks.csv' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [3]:
# Fill missing class labels according to row ranges in the documentation
df['class_filled'] = df['class']

df.loc[0:9425, 'class_filled'] = 'Benign'
df.loc[13717:25388, 'class_filled'] = 'DoS Attack'
df.loc[26363:38369, 'class_filled'] = 'Replay Attack'
df.loc[39344:45027, 'class_filled'] = 'Evil Twin'
df.loc[50502:54783, 'class_filled'] = 'FDI'

# Remove header rows if any
df = df[df['class_filled'].notna()]
df = df[df['class_filled'] != 'class']

print("Unique labels after filling/cleaning:", df['class_filled'].unique())

Unique labels after filling/cleaning: ['Benign' 'DoS Attack' 'Replay Attack' 'Evil Twin' 'FDI']


In [4]:
# Constants
RANDOM_STATE = 1082
TEST_SIZE = 0.2
BATCH_SIZE = 64
EPOCHS = 400
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# Data Preprocessing

# Drop columns not used as features
drop_cols = ['timestamp_c', 'class', 'class_filled']
# If timestamp_c or other cols don't exist, ignore errors
X = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

# Encode labels
le = LabelEncoder()
y = le.fit_transform(df['class_filled'])

print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
print("Unique classes after encoding:", np.unique(y))
num_classes = len(np.unique(y))

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

Label mapping: {'Benign': np.int64(0), 'DoS Attack': np.int64(1), 'Evil Twin': np.int64(2), 'FDI': np.int64(3), 'Replay Attack': np.int64(4)}
Unique classes after encoding: [0 1 2 3 4]


In [6]:
# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [7]:
# Define a simple MLP model for classification

class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.Sigmoid(),
            nn.Linear(128, 64),
            nn.Sigmoid(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.model(x)


model = MLP(X_train.shape[1], num_classes).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)

In [8]:
# Training loop

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_X.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch + 1}/{EPOCHS}, Loss: {epoch_loss:.4f}")

Epoch 1/400, Loss: 1.5441
Epoch 2/400, Loss: 1.4946
Epoch 3/400, Loss: 1.4675
Epoch 4/400, Loss: 1.4397
Epoch 5/400, Loss: 1.4093
Epoch 6/400, Loss: 1.3761
Epoch 7/400, Loss: 1.3401
Epoch 8/400, Loss: 1.3014
Epoch 9/400, Loss: 1.2609
Epoch 10/400, Loss: 1.2204
Epoch 11/400, Loss: 1.1811
Epoch 12/400, Loss: 1.1440
Epoch 13/400, Loss: 1.1101
Epoch 14/400, Loss: 1.0793
Epoch 15/400, Loss: 1.0515
Epoch 16/400, Loss: 1.0269
Epoch 17/400, Loss: 1.0060
Epoch 18/400, Loss: 0.9869
Epoch 19/400, Loss: 0.9700
Epoch 20/400, Loss: 0.9550
Epoch 21/400, Loss: 0.9414
Epoch 22/400, Loss: 0.9293
Epoch 23/400, Loss: 0.9178
Epoch 24/400, Loss: 0.9078
Epoch 25/400, Loss: 0.8987
Epoch 26/400, Loss: 0.8908
Epoch 27/400, Loss: 0.8834
Epoch 28/400, Loss: 0.8768
Epoch 29/400, Loss: 0.8707
Epoch 30/400, Loss: 0.8649
Epoch 31/400, Loss: 0.8594
Epoch 32/400, Loss: 0.8544
Epoch 33/400, Loss: 0.8502
Epoch 34/400, Loss: 0.8455
Epoch 35/400, Loss: 0.8414
Epoch 36/400, Loss: 0.8376
Epoch 37/400, Loss: 0.8338
Epoch 38/4

In [9]:
# Evaluation

model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(DEVICE)
        outputs = model(batch_X)
        _, preds = torch.max(outputs, 1)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(batch_y.numpy())

y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_labels)
accuracy = accuracy_score(y_true, y_pred)
print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))
print("\nClassification Report:")
print(classification_report(y_true, y_pred, zero_division=0))

Model Accuracy on Test Set: 68.96%
Confusion Matrix:
[[1700   14    0    0  171]
 [ 220 1433    0    0  681]
 [   0    0 1136    0    1]
 [   0    0  692  164    0]
 [ 267  628    0    0 1507]]

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.90      0.83      1885
           1       0.69      0.61      0.65      2334
           2       0.62      1.00      0.77      1137
           3       1.00      0.19      0.32       856
           4       0.64      0.63      0.63      2402

    accuracy                           0.69      8614
   macro avg       0.75      0.67      0.64      8614
weighted avg       0.72      0.69      0.67      8614

